In [ ]:
import copy, math

import numpy as np
import pickle as pkl
import numba as nb
import plotly.graph_objs as go
import plotly.offline as pyo
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from numba import njit
from scipy.stats import gaussian_kde

from matplotlib.colors import LogNorm, SymLogNorm, LinearSegmentedColormap, ListedColormap, to_rgb
from matplotlib.ticker import NullFormatter, LogLocator, SymmetricalLogLocator, FixedLocator, FixedFormatter, NullLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.patches import Rectangle

from fastkde.fastKDE import pdf as fastKDE_pdf
from scipy.integrate import cumulative_trapezoid

from scipy.fft import  fftn as  fftn
from scipy.fft import ifftn as ifftn
from scipy.ndimage import fourier_gaussian as fourier_gaussian
from scipy.ndimage import median_filter, grey_opening, grey_closing

from pathlib import Path
import re, cv2

---
---
---
---
---
---
---
---
---
---

In [ ]:
plots_path_0 = "../Plots/"

In [ ]:
sc = 80

In [ ]:
range_75_5 = np.arange(0,76,5)
range_75_5_tkss = [0, "", "", 15, "", "", 30, "", "", 45, "", "", 60, "", "", 75]

---

## Code functions

In [ ]:
layout = go.Layout(margin={'l': 0, 'r': 0, 'b': 0, 't': 0})

In [ ]:
def plot_3d_interactive(xyz_x, xyz_y, xyz_z, xyz_min=0, xyz_max=512, dd=0.1, show_planes=True):

    x_min = np.min(xyz_x); x_max = np.max(xyz_x)
    y_min = np.min(xyz_y); y_max = np.max(xyz_y)
    z_min = np.min(xyz_z); z_max = np.max(xyz_z)
    
    # Main data trace
    main_trace = go.Scatter3d(x=xyz_x, y=xyz_y, z=xyz_z,
                              mode='markers', marker=dict(size=5, color='blue', opacity=0.1),
                              name='Data Points')

    # Shadow trace
    shadow_trace = go.Scatter3d(x=[x + dd for x in xyz_x], y=[y + dd for y in xyz_y], z=[z - dd for z in xyz_z],
                                mode='markers', marker=dict(size=5, color='red', opacity=0.5),
                                name='Shadows')

    # Define axis range and tick values
    axis_range = [xyz_min, xyz_max]
    tickvals, tickvalss = set_ticks(xyz_min, xyz_max, log_lin=False, int_if_possible=True)


    
    # Create arrow traces for x, y, z axes
    arrow_length = xyz_max-xyz_min
    arrowhead_size = arrow_length/50

    # X-axis arrow (a line from origin to arrow_length along x)
    x_axis_trace = go.Scatter3d(x=[xyz_min, xyz_min+arrow_length], y=[xyz_min, xyz_min], z=[xyz_min, xyz_min],
                                mode='lines+markers', line=dict(color='black', width=4), marker=dict(size=[0, arrowhead_size], color='black', symbol='circle'),
                                showlegend=False)

    # Y-axis arrow
    y_axis_trace = go.Scatter3d(x=[xyz_min, xyz_min], y=[xyz_min, xyz_min+arrow_length], z=[xyz_min, xyz_min],
                                mode='lines+markers', line=dict(color='black', width=4), marker=dict(size=[0, arrowhead_size], color='black', symbol='circle'),
                                showlegend=False)

    # Z-axis arrow
    z_axis_trace = go.Scatter3d(x=[xyz_min, xyz_min], y=[xyz_min, xyz_min], z=[xyz_min, xyz_min+arrow_length],
                                mode='lines+markers', line=dict(color='black', width=4), marker=dict(size=[0, arrowhead_size], color='black', symbol='circle'),
                                showlegend=False)


    # Plane at x=0 (y-z plane)
    x0_plane = go.Surface(x=[[0, 0], [0, 0]], y=[[xyz_min, xyz_max], [xyz_min, xyz_max]], z=[[xyz_min, xyz_min], [xyz_max, xyz_max]],
                          colorscale=[[0, 'rgba(255, 165, 0, 0.2)'], [1, 'rgba(255, 165, 0, 0.2)']], showscale=False, opacity=0.2,
                          name='x=0 plane')

    # Plane at x=0 (y-z plane)
    xsize_plane = go.Surface(x=[[size-1, size-1], [size-1, size-1]], y=[[xyz_min, xyz_max], [xyz_min, xyz_max]], z=[[xyz_min, xyz_min], [xyz_max, xyz_max]],
                          colorscale=[[0, 'rgba(255, 165, 0, 0.2)'], [1, 'rgba(255, 165, 0, 0.2)']], showscale=False, opacity=0.2,
                          name='x=0 plane')

    # Plane at y=0 (x-z plane)
    y0_plane = go.Surface(x=[[xyz_min, xyz_max], [xyz_min, xyz_max]], y=[[0, 0], [0, 0]], z=[[xyz_min, xyz_min], [xyz_max, xyz_max]],
                          colorscale=[[0, 'rgba(0, 255, 0, 0.2)'], [1, 'rgba(0, 255, 0, 0.2)']], showscale=False, opacity=0.2,
                          name='y=0 plane')

    # Plane at y=0 (x-z plane)
    ysize_plane = go.Surface(x=[[xyz_min, xyz_max], [xyz_min, xyz_max]], y=[[size-1, size-1], [size-1, size-1]], z=[[xyz_min, xyz_min], [xyz_max, xyz_max]],
                          colorscale=[[0, 'rgba(0, 255, 0, 0.2)'], [1, 'rgba(0, 255, 0, 0.2)']], showscale=False, opacity=0.2,
                          name='y=0 plane')

    # Plane at z=0 (x-y plane)
    z0_plane = go.Surface(x=[[xyz_min, xyz_max], [xyz_min, xyz_max]], y=[[xyz_min, xyz_min], [xyz_max, xyz_max]], z=[[0, 0], [0, 0]],
                          colorscale=[[0, 'rgba(0, 0, 255, 0.2)'], [1, 'rgba(0, 0, 255, 0.2)']], showscale=False, opacity=0.2,
                          name='z=0 plane')

    # Plane at z=0 (x-y plane)
    zsize_plane = go.Surface(x=[[xyz_min, xyz_max], [xyz_min, xyz_max]], y=[[xyz_min, xyz_min], [xyz_max, xyz_max]], z=[[size-1, size-1], [size-1, size-1]],
                          colorscale=[[0, 'rgba(0, 0, 255, 0.2)'], [1, 'rgba(0, 0, 255, 0.2)']], showscale=False, opacity=0.2,
                          name='z=0 plane')

    
    # Define the layout
    layout = go.Layout(scene=dict(xaxis=dict(title='x [cMpc/h]',
                                             range=axis_range, tickvals=tickvals, ticktext=tickvalss, ticks='outside', showticklabels=True,
                                             showbackground=False, showgrid=False, zeroline=False, showline=True),
                                  yaxis=dict(title='y [cMpc/h]',
                                             range=axis_range, tickvals=tickvals, ticktext=tickvalss, ticks='outside', showticklabels=True,
                                             showbackground=False, showgrid=False, zeroline=False, showline=True),
                                  zaxis=dict(title='z [cMpc/h]',
                                             range=axis_range, tickvals=tickvals, ticktext=tickvalss, ticks='outside', showticklabels=True,
                                             showbackground=False, showgrid=False, zeroline=False, showline=True),
                                  aspectmode='cube'), showlegend=True)


    if show_planes: fig = go.Figure(data=[shadow_trace, main_trace, x_axis_trace, y_axis_trace, z_axis_trace, x0_plane, xsize_plane, y0_plane, ysize_plane, z0_plane, zsize_plane], layout=layout)
    else:           fig = go.Figure(data=[shadow_trace, main_trace, x_axis_trace, y_axis_trace, z_axis_trace, x0_plane,              y0_plane,              z0_plane             ], layout=layout)
    pyo.plot(fig)

---
---
---

In [ ]:
cls = ['r', 'b']
pixels_bin_x = 10
pixels_bin_z = 121

In [ ]:
def hist_pixels_plot(list_plot, lmin1, lmax1, tit, save_name, legend_labels_Zs, xlbl1, ylbl1="Counts", tks_scale_y=1, zero_round=0.01, equality_02=2, int_if_possible=False, showoff=False, log_lin_x=True, log_lin_y=False, zero_log_min=1/2, pixels_bin_x=10, pixels_bin_z=121, all3=False, get_a_stable_exponent_y=False, save_true=True, all3_all=False, return_lengths=False):
    
    lzP = len(list_plot)
    
    
    ### pixels    
    pixels = [[0.1 for i in range(pixels_bin_x*(lzP))] for j in range(pixels_bin_z)]
    
    ### going
    going = [0,0,0]
    list_plot_upper     = []; list_plot_lower     = []; list_plot_middle     = []
    len_list_plot_upper = []; len_list_plot_lower = []; len_list_plot_middle = []
    for i in list_plot:
        list_plot_upper.append([]); list_plot_lower.append([]); list_plot_middle.append([])
        for j in i:
            # We separate them like such for the case where we might have all elements in a linerar x-axis in a closed interval including 0
            #    on just one side of it: [0, 123] or [-123, 0].
            if   equality_02 == 0:
                if   j <= -zero_round: going[0] = 1; list_plot_lower[ -1].append(np.abs(j))
                elif j <   zero_round: going[1] = 1; list_plot_middle[-1].append(       j )
                else:                  going[2] = 1; list_plot_upper[ -1].append(       j )
            elif equality_02 == 2:
                if j   < -zero_round: going[0] = 1; list_plot_lower[ -1].append(np.abs(j))
                elif j <  zero_round: going[1] = 1; list_plot_middle[-1].append(       j )
                else:                 going[2] = 1; list_plot_upper[ -1].append(       j )
        len_list_plot_lower.append( len(list_plot_lower[ -1]))
        len_list_plot_middle.append(len(list_plot_middle[-1]))
        len_list_plot_upper.append( len(list_plot_upper[ -1]))
    len_list_plot_all = [len_list_plot_lower, len_list_plot_middle, len_list_plot_upper]
    
    if going[0] == going[2] == 1: going[1] = 1   # If we display the negative and positive intervals, we show the middle one too.
    
    # Despite our data set, we must respect our set limits.
    start_round = zero_round; end_round = -zero_round
    if lmin1 >   -zero_round: going[0] = 0; start_round =  zero_round
    if lmin1 >=   zero_round: going[1] = 0; start_round =  lmin1
    if lmax1 <    zero_round: going[2] = 0; end_round   = -zero_round
    if lmax1 <=  -zero_round: going[1] = 0; end_round   =  lmax1
    
    ### list_plots & pbz
    list_plots  = []
    list_plotss = [list_plot_lower, list_plot_middle, list_plot_upper]
    for i in range(3):
        if going[i] == 1: list_plots.append(list_plotss[i])
    
    
    ### bins_x
    if going[1] == 1:
        lpm = [i for i in list_plot_middle if (i != [])]
        # Now, in case we have upper and lower but no middle, we must set default mdl_min and mdl_max:
        mdl_min = -zero_round; mdl_max = zero_round
        
        if lpm != []:
            mdl_min1 = np.min([np.min(i) for i in lpm])
            mdl_max1 = np.max([np.max(i) for i in lpm])
            if mdl_min1 < lmin1: mdl_min1 = lmin1
            if mdl_max1 > lmax1: mdl_max1 = lmax1
        
        if going == [1,1,1]: mdl_min = -zero_round; mdl_max = zero_round
        if going == [1,1,0]: mdl_min = -zero_round; mdl_max = mdl_max1
        if going == [0,1,1]: mdl_min = mdl_min1;    mdl_max = zero_round
        if going == [0,1,0]: mdl_min = mdl_min1;    mdl_max = mdl_max1
    
    all_3_ranges = [0,0,0]; pbz = [0,0,0]
    ii = 0
    if going == [1,1,1]:
        for i in range(3):
            if i == 0:
                if log_lin_x: all_3_ranges[0] = np.log10(np.abs(lmin1)) - np.log10(np.abs(end_round))
                else:         all_3_ranges[0] = np.abs(lmin1) - np.abs(end_round)
            if i == 1:        pbz[1] = 1+1
            if i == 2:
                if log_lin_x: all_3_ranges[2] = np.log10(np.abs(lmax1)) - np.log10(np.abs(start_round))
                else:         all_3_ranges[2] = np.abs(lmax1) - np.abs(start_round)
        
        pbz_0 = int(round((pixels_bin_z-1) * all_3_ranges[0]/(all_3_ranges[0]+all_3_ranges[2]),0))+1
        pbz[0] = pbz_0
        pbz[2] = pixels_bin_z-(2-1)-(pbz_0-1)+1
    
    
    #if   going == [1,1,1]: pbz = [(pixels_bin_z-1)/2+1, 1+1, (pixels_bin_z-1)/2+1]
    if   going == [0,1,1]: pbz = [1+1, (pixels_bin_z-1)+1]
    elif going == [1,1,0]: pbz = [(pixels_bin_z-1)+1, 1+1]
    elif going == [1,0,0]: pbz = [pixels_bin_z+1]
    elif going == [0,0,1]: pbz = [pixels_bin_z+1]
    elif going == [0,1,0]: pbz = [1+1]
    elif going == [0,0,0]: pbz = []
    pbz = [int(i) for i in pbz if i!=0]
    
    bins_x = []
    ii = 0
    for i in range(3):
        if going[i] == 1:
            if log_lin_x:
                if i == 0: bins_x.append(10**(np.linspace(np.log10(np.abs(end_round)), np.log10(np.abs(lmin1)), pbz[ii])))
                if i == 1: bins_x.append(np.array([mdl_min, mdl_max]))
                if i == 2: bins_x.append(10**(np.linspace(np.log10(np.abs(start_round)), np.log10(np.abs(lmax1)), pbz[ii])))
            else:
                if i == 0: bins_x.append(np.linspace(np.abs(end_round), np.abs(lmin1), pbz[ii]))
                if i == 1: bins_x.append(np.array([mdl_min, mdl_max]))
                if i == 2: bins_x.append(np.linspace(np.abs(start_round), np.abs(lmax1), pbz[ii]))
            ii += 1
                
    
    bins_x_all = []
    for i in bins_x:
        for j in i: bins_x_all.append(j)
    
            
    plt_lims = []
    if   going[0] == 1: plt_lims.append(lmin1)
    elif going[1] == 1: plt_lims.append(mdl_min)
    elif going[2] == 1: plt_lims.append(start_round)
    
    if   going[2] == 1: plt_lims.append(lmax1)
    elif going[1] == 1: plt_lims.append(mdl_max)
    elif going[0] == 1: plt_lims.append(end_round)
    
##################################################################################################################################
    
    fin_pixel_val_y_max_list = []; fin_pixel_val_x_max_list = []; fin_pixel_ind_x_max_list = []
    fin_pixel_val_y_min_list = []; fin_pixel_val_x_min_list = []; fin_pixel_ind_x_min_list = []
    
    fin_pixel_val_y_max_list_all3 = []; fin_pixel_val_x_max_list_all3 = []; fin_pixel_ind_x_max_list_all3 = []; fin_pixel_going_max_list_all3 = []
    fin_pixel_val_x_min_list_all3 = []; fin_pixel_val_y_min_list_all3 = []; fin_pixel_ind_x_min_list_all3 = []; fin_pixel_going_min_list_all3 = []
        
    fig, axs = plt.subplots(1, 3, figsize=(14,6), dpi=400, sharey=True, gridspec_kw={'width_ratios': [going[0],going[1]/10,going[2]]})
    plt.suptitle(tit)
    fig.subplots_adjust(wspace=0)
    
    nn1 = 0; io = -1
    for i in range(lzP):
        if i in [0, lzP-1]: io += 1
        iii = -1; been = 0
        
        # MEAN
        pixel_val_x_mean_list = [0,0,0]
        pixel_val_x_mean_list_all3 = 0
        
        # ALL3 MAX
        pixel_val_y_max_all3 = -np.inf
        pixel_val_y_max_list_all3 = []; pixel_val_x_max_list_all3 = []; pixel_ind_x_max_list_all3 = []; pixel_going_max_list_all3 = []
        
        # ALL3 MIN
        pixel_val_y_min_all3 = np.inf
        pixel_val_y_min_list_all3 = []; pixel_val_x_min_list_all3 = []; pixel_ind_x_min_list_all3 = []; pixel_going_min_list_all3 = []
        
        # EACH3 MAX
        pixel_val_y_max      = [-np.inf, -np.inf, -np.inf]
        pixel_val_y_max_list = [[], [], []]; pixel_val_x_max_list = [[], [], []]; pixel_ind_x_max_list = [[], [], []]
        
        # EACH3 MIN
        pixel_val_y_min      = [np.inf, np.inf, np.inf]
        pixel_val_y_min_list = [[], [], []]; pixel_val_x_min_list = [[], [], []]; pixel_ind_x_min_list = [[], [], []]
        
        pixel_val_x_mean_list_all3 = np.mean(list_plot[i])

        for ii in range(3):
            if going[ii] == 1:
                iii += 1
                lp_i = list_plots[iii][i]
                bx_i = bins_x[iii]
                
                if i in [0, lzP-1]:
                    by, bx, _ = axs[ii].hist(lp_i, bins=bx_i, alpha=0.3, label='Z='+str(legend_labels_Zs[i]), color=cls[io])
                    axs[ii].hist(lp_i, bins=bx_i, alpha=0.6, color=cls[io], histtype='step', zorder=9 if i == 0 else 3)
                else:
                    by, bx, _ = axs[ii].hist(lp_i, bins=bx_i, alpha=0.3)
                
                for j1 in range(pbz[iii]-1):
                    if by[j1] != 0:   # Remember, we keep min in each pixel to be 0.1 so we don't mess up the log colorbar.
                        for j2 in range(pixels_bin_x):
                            if ii == 2:           pixels[int(pbz[-1]-2-j1)][int(i*pixels_bin_x+j2)] = by[j1]
                            if ii == 1:
                                if going[2] == 1: pixels[int(pbz[-1]-1)][int(i*pixels_bin_x+j2)] = by[j1]
                                else:             pixels[0][int(i*pixels_bin_x+j2)] = by[j1]
                            if ii == 0:           pixels[pixels_bin_z+int(-pbz[0]+1+j1)][int(i*pixels_bin_x+j2)] = by[j1]
                
                pixel_val_x_mean_list[ii] = np.mean(lp_i)
                
                for byi in range(len(by)):
                    if ii == 2:           byiyi = pixels_bin_z-pbz[-1]+1+byi
                    if ii == 1:
                        if going[2] == 1: byiyi = pixels_bin_z-pbz[-1]
                        else:             byiyi = pixels_bin_z-1
                    if ii == 0:           byiyi = pbz[0]-2-byi +1 # +1 cuz pixels
                    byiyi += 0.5
                    
                    # ALL3 MAX
                    if pixel_val_y_max_all3 <  by[byi]: pixel_val_y_max_list_all3 = []; pixel_val_x_max_list_all3 = []; pixel_ind_x_max_list_all3 = []; pixel_going_max_list_all3 = []
                    if pixel_val_y_max_all3 <= by[byi]: pixel_val_y_max_all3 = by[byi]; pixel_val_y_max_list_all3.append(by[byi]); pixel_val_x_max_list_all3.append((bx[byi]+bx[byi+1])/2); pixel_ind_x_max_list_all3.append(byiyi); pixel_going_max_list_all3.append(ii)
                    
                    # ALL3 MIN
                    if pixel_val_y_min_all3 > by[byi]: pixel_val_y_min_list_all3 = []; pixel_val_x_min_list_all3 = []; pixel_ind_x_min_list_all3 = []; pixel_going_min_list_all3 = []
                    if pixel_val_y_min_all3 >= by[byi]: pixel_val_y_min_all3 = by[byi]; pixel_val_y_min_list_all3.append(by[byi]); pixel_val_x_min_list_all3.append((bx[byi]+bx[byi+1])/2); pixel_ind_x_min_list_all3.append(byiyi); pixel_going_min_list_all3.append(ii)
                    
                    # EACH3 MAX
                    if pixel_val_y_max[ii] <  by[byi]: pixel_val_y_max_list[ii] = []; pixel_val_x_max_list[ii] = []; pixel_ind_x_max_list[ii] = []
                    if pixel_val_y_max[ii] <= by[byi]: pixel_val_y_max[ii] = by[byi]; pixel_val_y_max_list[ii].append(by[byi]); pixel_val_x_max_list[ii].append((bx[byi]+bx[byi+1])/2); pixel_ind_x_max_list[ii].append(byiyi)
                    
                    # EACH3 MIN
                    if pixel_val_y_min[ii] >  by[byi]: pixel_val_y_min_list[ii] = []; pixel_val_x_min_list[ii] = []; pixel_ind_x_min_list[ii] = []
                    if pixel_val_y_min[ii] >= by[byi]: pixel_val_y_min[ii] = by[byi]; pixel_val_y_min_list[ii].append(by[byi]); pixel_val_x_min_list[ii].append((bx[byi]+bx[byi+1])/2); pixel_ind_x_min_list[ii].append(byiyi)
                    
                if nn1 < pixel_val_y_max_all3: nn1 = pixel_val_y_max_all3
                
                
        if i in [0, lzP-1]:
            
            if all3 or all3_all:
                if    zero_round <= pixel_val_x_mean_list_all3: iilp = 2
                elif -zero_round <  pixel_val_x_mean_list_all3: iilp = 1
                else: iilp = 0
                
                tk0 = latex_float(pixel_val_x_mean_list_all3)
                axs[iilp].axvline(x=pixel_val_x_mean_list_all3, color=cls[io], linestyle='dashdot', alpha=0.6, label='Mean_all = '+tk0)
                if not all3_all:
                    for i5 in range(len(pixel_val_x_max_list_all3)):
                        iilp = pixel_going_max_list_all3[i5]
                        if i5 == 0:
                            tk0 = latex_float(pixel_val_x_max_list_all3[i5])
                            axs[iilp].axvline(x=pixel_val_x_max_list_all3[i5], color=cls[io], linestyle='dashdot', alpha=0.6, label='Max   = '+tk0)
                        else:
                            axs[iilp].axvline(x=pixel_val_x_max_list_all3[i5], color=cls[io], linestyle='dashdot', alpha=0.6)
                
            if not all3:
                for ii in range(3):
                    if ii in [0, 2]:
                        sgn = 1
                        if ii == 0: sgn = -1
                        tk0 = latex_float(sgn*pixel_val_x_mean_list[ii])
                        axs[ii].axvline(x=pixel_val_x_mean_list[ii], color=cls[io], linestyle='dashed', alpha=0.6, label='Mean = '+tk0)
                        for i5 in range(len(pixel_val_x_max_list[ii])):
                            if i5 == 0:
                                tk0 = latex_float(sgn*pixel_val_x_max_list[ii][i5])
                                axs[ii].axvline(x=pixel_val_x_max_list[ii][i5], color=cls[io], linestyle='solid', alpha=0.6, label='Max   = '+tk0)
                            else:
                                axs[ii].axvline(x=pixel_val_x_max_list[ii][i5], color=cls[io], linestyle='solid')

        fin_pixel_val_y_max_list.append(pixel_val_y_max_list); fin_pixel_val_x_max_list.append(pixel_val_x_max_list); fin_pixel_ind_x_max_list.append(pixel_ind_x_max_list)
        fin_pixel_val_y_min_list.append(pixel_val_y_min_list); fin_pixel_val_x_min_list.append(pixel_val_x_min_list); fin_pixel_ind_x_min_list.append(pixel_ind_x_min_list)
        
        fin_pixel_val_y_max_list_all3.append(pixel_val_y_max_list_all3); fin_pixel_val_x_max_list_all3.append(pixel_val_x_max_list_all3); fin_pixel_ind_x_max_list_all3.append(pixel_ind_x_max_list_all3); fin_pixel_going_max_list_all3.append(pixel_going_max_list_all3)
        fin_pixel_val_y_min_list_all3.append(pixel_val_y_min_list_all3); fin_pixel_val_x_min_list_all3.append(pixel_val_x_min_list_all3); fin_pixel_ind_x_min_list_all3.append(pixel_ind_x_min_list_all3); fin_pixel_going_min_list_all3.append(pixel_going_min_list_all3)
        
    plt.plot([],[],alpha=0,label=' ')
        
    if log_lin_x:
        if going[0] == 1: axs[0].set_xscale('log')
        if going[2] == 1: axs[2].set_xscale('log')
    
    for ii in range(3):
        if going[ii] == 0:
            axs[ii].set_xticks([],[])    
        else:
            if ii == 1:
                xtks = [mdl_min, mdl_max]
                if going == [1,1,1]: xtkss = ['', '']
                if going == [1,1,0]: xtkss = ['', latex_float(mdl_max)]
                if going == [0,1,1]: xtkss = [latex_float(mdl_min), '']
                if going == [0,1,0]: xtkss = [str(-zero_round), str(zero_round)]
            
            if ii == 0:
                xtks, xtkss0 = set_ticks(-end_round, -lmin1, log_lin=log_lin_x)
                xtks0, xtkss = set_ticks(lmin1, end_round, int_base_if_possible=True)
                xtkss = xtkss[::-1]
            
            if ii == 2:
                xtks, xtkss = set_ticks(start_round, lmax1, log_lin=log_lin_x, int_base_if_possible=True)
            
            axs[ii].set_xticks(xtks, xtkss)
    
    
    for ii in range(3): axs[ii].grid()
    
    if going == [1,1,1]:
        axs[0].spines['right'].set_visible(False)
        axs[1].spines['left' ].set_visible(False)
        axs[1].spines['right'].set_visible(False)
        axs[2].spines['left' ].set_visible(False)
    if going == [1,1,0]:
        axs[0].spines['right'].set_visible(False)
        axs[1].spines['left' ].set_visible(False)
    if going == [0,1,1]:
        axs[1].spines['right'].set_visible(False)
        axs[2].spines['left' ].set_visible(False)
    
    axs[0].set_ylabel(ylbl1)
#    for i in range(3):
#        if going[i] == 1: first_true_axis = i; break
    first_true_axis = 0

    zero_log = 0
    if log_lin_y:
        zero_log = zero_log_min
        for i in range(3): axs[i].set_yscale("log")
    if get_a_stable_exponent_y:
        if tks_scale_y == 1:
            ytks, ytkss, stbl_exp = set_ticks(zero_log, nn1,             log_lin=log_lin_y, int_if_possible=int_if_possible, auto_stable_exponent=True)
        else:
            ytks, ytkss, stbl_exp = set_ticks(zero_log, nn1*tks_scale_y, log_lin=log_lin_y, int_if_possible=int_if_possible, auto_stable_exponent=True)
            ytks = list(np.array(ytks) / tks_scale_y)
        xmin, xmax = axs[first_true_axis].get_xlim()
        x_pos = xmax*0.8
        y_pos = axs[first_true_axis].get_ylim()[1]*1.01
        if stbl_exp != 0: axs[first_true_axis].text(x_pos, y_pos, r"$\times 10^{{{0}}}$".format(int(stbl_exp)), ha='center')
    else:
        if tks_scale_y == 1:
            ytks, ytkss = set_ticks(zero_log, nn1,             log_lin=log_lin_y, int_if_possible=int_if_possible, auto_stable_exponent=False)
        else:
            ytks, ytkss = set_ticks(zero_log, nn1*tks_scale_y, log_lin=log_lin_y, int_if_possible=int_if_possible, auto_stable_exponent=False)
            ytks = list(np.array(ytks) / tks_scale_y)
            
    
    if ytks[-1]/ytks[-2] < 1.1:
        ytks  = ytks[ :-2]+ytks[ -1:]
        ytkss = ytkss[:-2]+ytkss[-1:]
    if zero_log == zero_log_min:
        ytkss[0] = "0"
    axs[first_true_axis].set_yticks(ytks, ytkss)

    
    for ii in range(3):
        if going[ii] == 1:
            if ii == 1: lim_xtks = [mdl_min, mdl_max]
            if ii == 0: lim_xtks = [-end_round, -lmin1]
            if ii == 2: lim_xtks = [start_round, lmax1]
            axs[ii].set_xlim(lim_xtks)
    if (going[0] == 1) and (going[2] == 1): lim_xtks = [lmin1, lmax1]
    
    axs[0].invert_xaxis()
    if going in [[1,1,1], [0,1,0]]: axs[1].set_xlabel('\n'+xlbl1)
    if going in [[1,1,0], [1,0,0]]: axs[0].set_xlabel('\n'+xlbl1)
    if going in [[0,1,1], [0,0,1]]: axs[2].set_xlabel('\n'+xlbl1)
    
    if going in [[1,0,0], [1,1,0], [1,1,1]]: axs[0].legend(loc=2)
    if going in [[0,0,1], [0,1,1], [1,1,1]]: axs[2].legend(loc=1)
    
    if save_true: plt.savefig(save_name, bbox_inches='tight')
    if showoff:   plt.show()
    else:         plt.close()

    if not return_lengths:
        if not all3: return pixels, lim_xtks, fin_pixel_val_y_max_list, fin_pixel_val_x_max_list, fin_pixel_ind_x_max_list, fin_pixel_val_y_min_list, fin_pixel_val_x_min_list, fin_pixel_ind_x_min_list, list_plots, bins_x, going, plt_lims
        else:        return pixels, lim_xtks, fin_pixel_val_y_max_list_all3, fin_pixel_val_x_max_list_all3, fin_pixel_ind_x_max_list_all3, fin_pixel_going_max_list_all3, fin_pixel_val_y_min_list_all3, fin_pixel_val_x_min_list_all3, fin_pixel_ind_x_min_list_all3, fin_pixel_going_min_list_all3, list_plots, bins_x, going, plt_lims
    else:
        if not all3: return len_list_plot_all, pixels, lim_xtks, fin_pixel_val_y_max_list, fin_pixel_val_x_max_list, fin_pixel_ind_x_max_list, fin_pixel_val_y_min_list, fin_pixel_val_x_min_list, fin_pixel_ind_x_min_list, list_plots, bins_x, going, plt_lims
        else:        return len_list_plot_all, pixels, lim_xtks, fin_pixel_val_y_max_list_all3, fin_pixel_val_x_max_list_all3, fin_pixel_ind_x_max_list_all3, fin_pixel_going_max_list_all3, fin_pixel_val_y_min_list_all3, fin_pixel_val_x_min_list_all3, fin_pixel_ind_x_min_list_all3, fin_pixel_going_min_list_all3, list_plots, bins_x, going, plt_lims

In [ ]:
def color_double_hist_pixels_plot(pixels, x_ticks_list, limits_y, fin_pixel_val_y_max_list_all3_i, fin_pixel_val_x_max_list_all3_i, fin_pixel_ind_x_max_list_all3_i, tit, ylabel, save_name, cbar_label, volume_density, zero_round=0.01, pixels_bin_z=121, legend_loc=1, log_lin_y=True, float_g_y=False, showoff=True, print_all3_lines=False, int_if_possible=True, prend=True, plot_counts=True, norm_the_pixels=True):



    plt.figure(figsize=(10,5), dpi=300)
    plt.title(tit)
    
    pixels_normed = copy.deepcopy(pixels)
    if norm_the_pixels:
        all_norms = []
        for i0 in range(int(len(pixels[0])/10)): #each redshift - 14
            zbin = []
            for i1 in range(len(pixels)): #121
                zbin.append(pixels[i1][i0*10])
            norm_z = np.sum(zbin)
            all_norms.append(norm_z)
            for i1 in range(len(pixels)): #121
                for i2 in range(10):
                    pixels_normed[i1][i0*10+i2] /= norm_z
    
    mask = (np.array(pixels) == 0.1)
    pixels_plot = np.ma.masked_where(mask, pixels_normed)
    
    cmap = plt.cm.inferno.copy()
    cmap.set_bad("black")
    
    vmini = np.min(pixels_plot); vmaxi = np.max(pixels_plot)
    imm = plt.imshow(pixels_plot, norm=LogNorm(vmin=vmini, vmax=vmaxi), cmap=cmap, extent=[0.0-0.5, len(fin_pixel_ind_x_max_list_all3_i)-0.5, 1, pixels_bin_z], aspect='auto')
    
    
    for i in range(len(fin_pixel_ind_x_max_list_all3_i)):
        if i == 0:
            if plot_counts: lbl_r = 'Max '+volume_density+' bin(s) (value & count)'
            else:           lbl_r = 'Max '+volume_density+' bin(s) (value)'
        else: lbl_r = ''
    
        clk = 'green'
    
        print_kopi = 0
        kopi = 1
        if not print_all3_lines: kopi = 0
    
        for jj in range(len(fin_pixel_ind_x_max_list_all3_i[i])):
            if (len(fin_pixel_ind_x_max_list_all3_i[i]) != 3) or (jj != 1):
                for jjj in range(len(fin_pixel_ind_x_max_list_all3_i[i][jj])):
                    if not print_all3_lines: kopi += 1
                    print_kopi += 1
                    for l in range(-1,2):
                        plt.scatter(i-l/3, 0.5+fin_pixel_ind_x_max_list_all3_i[i][jj][jjj], color=clk, marker='_', s=160, lw=0.5, alpha=1)
                        if (l == 0) and (kopi == 1):
                            if print_kopi == 1:
                                plt.scatter(i, 0.5+fin_pixel_ind_x_max_list_all3_i[i][jj][jjj], color=clk, marker='_', s=160, lw=0.5, label=lbl_r, alpha=0.8)
                            else:
                                plt.scatter(i, 0.5+fin_pixel_ind_x_max_list_all3_i[i][jj][jjj], color=clk, marker='_', s=160, lw=0.5, alpha=0.8)
                            plt.text(i-0.45, 0.5+fin_pixel_ind_x_max_list_all3_i[i][jj][jjj]+0.6, latex_float(fin_pixel_val_x_max_list_all3_i[i][jj][0]), fontsize=5, c=clk, alpha=0.8)
                            if plot_counts:
                                plt.text(i-0.45, 0.5+fin_pixel_ind_x_max_list_all3_i[i][jj][jjj]-2.1, int(fin_pixel_val_y_max_list_all3_i[i][jj][jjj]), fontsize=5, c=clk, alpha=0.8)
                            plt.axvline(x=i+0.5, c='black', lw=0.5)
    
    cbar = plt.colorbar(imm, orientation='vertical')
    tks, tkss = set_ticks(vmini, vmaxi, log_lin=True, float_asitis_ends=prend, int_if_possible=int_if_possible)
    cbar.ax.set_yticks(tks, tkss)
    cbar.set_label(cbar_label, rotation=270, labelpad=15)
    
    plt.xlabel('z')
    plt.ylabel(ylabel)
    
    plt.xticks(range(len(fin_pixel_ind_x_max_list_all3_i)), [x_ticks_list[i] for i in range(len(fin_pixel_ind_x_max_list_all3_i))])
    
    ymin = limits_y[0]; ymax = limits_y[1]
    ytks0, ytkss = set_ticks(ymin, ymax, log_lin=log_lin_y, float_g=float_g_y, log_min=zero_round)
    if log_lin_y:
        if np.sign(ytks0[0]) * np.sign(ytks0[-1]) < 0:
            arg0 = np.argmin([np.abs(_) for _ in ytks0])-1
            ytks = []
            for i9 in [1+int((np.log10(np.abs(y))-np.log10(np.abs(ytks0[0])))/((np.log10(np.abs(ytks0[arg0]))-np.log10(np.abs(ytks0[0])))/(pixels_bin_z-2)*2)) for y in ytks0[:arg0+1]]: ytks.append(i9)
            ytks.append((pixels_bin_z-1)/2)
            for i9 in [(pixels_bin_z-1)/2+int((np.log10(y)-np.log10(ytks0[arg0+2]))/((np.log10(ytks0[-1]) - np.log10(np.abs(ytks0[arg0+2])))/(pixels_bin_z-1)*2)) for y in ytks0[arg0+2:]]: ytks.append(i9)
            for i0 in np.where(np.abs(np.array(ytks0)) == zero_round)[0]: ytkss[i0] = ""
        else:
            ytks = [1+int((np.log10(y)-np.log10(ytks0[0]))/((np.log10(ytks0[-1]) - np.log10(ytks0[0]))/(pixels_bin_z-1))) for y in ytks0]
    else:
        ytks = [1+int((y-ytks0[0])/((ytks0[-1] - ytks0[0])/(pixels_bin_z-1))) for y in ytks0]
    plt.yticks(ytks, ytkss)
    
    plt.legend(loc=legend_loc)
    plt.savefig(save_name, bbox_inches='tight')
    if showoff: plt.show()
    else:       plt.close()

---
---
---

I just wanted to make the numbers and their powers exactly how I wanted them and exactly which ones I wanted based on some conditions and automate the process (as to make this code work for any other data sets) and I admit I went a bit overkill..

In [ ]:
def set_ticks(vmin, vmax, log_lin=True, clean=True, d_tks_notclean=9999, number_of_tks_notclean=2, prioritize0_lin_notclean=True, factor_cut=1/15, factor_cut_2intervals=True,
              number_of_tks_clean_min=1, number_of_tks_clean_max=10, pick_clean_biggest_order_over_mean=True,
              whole_or_half=False, whole_only=False, set_factor=9999, set_base_power_10=9999, center0=0, log_min=0.01, auto_log_min=True, auto_log_min_equalityallowed=False,
              log_middle_include_0=True, only_tick_at_log_min=False, inside_log_use=False, ignore_tks_in_tkss = True,
              auto_stable_exponent=False, float_asitis_ends=False, g=2, stable_exponent=9999, pure_1=True, pure_0=True, float_asitis=False, float_g=False, int_if_possible=False, int_base_if_possible=True, exponent_0=False, clean_powers=True):

    
    '''
    ### INPUT
    vmin                               - the smallest value in the interval we set ticks
    vmax                               - the largest value in the interval we set ticks
    log_lin                            - whether we want the interval to be split logaritmically (alternatively it is lineraly split)
    clean                              - whether we want the interval to be split into the lowest power of 10 times factors or not
    d_tks_notclean                     - a set d_tks value used inside lin scale clean=False
    number_of_tks_notclean             - number of ticks (aside from the ends) when using lin scale and clean=False or inside the log scale when referring to the lin one
    prioritize0_lin_notclean           - prioritize keeping 0 in the ticks if vmin<0<vmax in the lin scale and clean=False over a tick next to it that might ve below
                                            the factor_cut threshold
    factor_cut                         - the factor of the total interval length that cuts-out any tick in that range next to vmin, vmax or log_min
    factor_cut_2intervals              - in the case we have a log scale interval that passes through 0, we may chose to double this interval (on each side) if both sides' intervals contain
                                            at least two elements
    number_of_tks_clean_min            - the minimum number of tks to obtain
    number_of_tks_clean_max            - the maximum number of tks to obtain
    pick_clean_biggest_order_over_mean - if we want to pick the biggest factor of the resulting tks lists that respect number_of_tks_clean_min and number_of_tks_clean_max
    set_factor                         - a set factor
    whole_or_half                      - use factors [0.1, 0.5, 0.01, 0.05], compared otherwise to [0.1, 0.2, 0.25, 0.3, 0.5, 0.01, 0.02, 0.025, 0.03, 0.05]
    whole_only                         - use factors [0.1, 0.01]
    set_base_power_10                  - a set smallest power of 10 from which we use factors
    center0                            - in the lin scale, this is the center value from which we add/subtrast d_tks
    log_min                            - the minimum value between vmin and vmax and 0
    auto_log_min                       - if log_min does not offer at least one power order, then it gets automatically changed to a value that does
    auto_log_min_equalityallowed       - auto_log_min does not occur if (np.abs(vmin) == log_min) and (np.abs(vmax) == log_min) and this makes sure the new log_min is not == vmin or vmax
    log_middle_include_0               - in the log scale include 0 if vmin<0 and vmax>0
    only_tick_at_log_min               - if we use log scale and 0 is a tick, we may chose to not display log_min ticks as latex (so just keep the ticks)
    inside_log_use                     - used inside this code for self-reference
    ignore_tks_in_tkss                 - if we have max and min values in the log scale that are within one order of magnitude, we may chose to not display the values between
    
    auto_stable_exponent               - see documentaiton of latex_float/latex_float_list
    float_asitis_ends                  - ...
    g                                  - ...
    stable_exponent                    - ...
    pure_1                             - ...
    pure_0                             - ...
    float_asitis                       - ...
    float_g                            - ...
    int_if_possible                    - ...
    int_base_if_possible               - ...
    exponent_0                         - ...
    clean_powers                       - ...
    '''
    
    vmin = round(vmin,11); vmax = round(vmax,11)
    
    if vmin == vmax:
        print('Equal ends... bruh...')
        return
    
    # list of values; list of respective floats - we sort them at the end
    tks = [vmin, vmax]

    tks_ignored_in_tkss = []
    intervals = [[vmin, vmax]]
    
    if not log_lin:
        
        if not clean:
            if d_tks_notclean != 9999:
                d_tks = d_tks_notclean
                i = 1; new_tick = round(vmin+i*d_tks,14)
                while new_tick < vmax:
                    tks.append(new_tick)
                    i += 1; new_tick = round(vmin+i*d_tks,14)
                    
            else:
                d_tks = np.abs(vmax-vmin)/(number_of_tks_notclean+1)
                for i in range(1,number_of_tks_notclean+1): tks.append(round(vmin+i*d_tks,14))
            if (vmin < 0 < vmax) and prioritize0_lin_notclean and (0 not in tks):
                tks = [_ for _ in tks if np.abs(_) >= (vmax-vmin)*factor_cut]
                tks.append(0)
            tks.sort()
    
        else:
            # The idea is:
            # [vmin,vmax] = [-21, 83]  ->  base_power_10 = 10
            # factor = 0.2
            # center0 = 0.3
            # result: [-21.0, -19.7, -17.7, ... 80.3, 82.3, 83.0]

            # Now, say we use whole_only=True and for simplicity center0=0... then we get:
            # [-21, -20, -19, ..., 81, 82, 83]
            # [-21.0, -20.9, 20.8, ..., 82.7, 82.9, 83.0]
            # Of course, the first one is more practical... so why do the second? Because of this case:
            # [vmin,vmax] = [13,112]  ->  base_power_10 = 100
            # [13, 100, 112]
            # [13, 20, 30, ..., 90, 100, 110, 112]
            
            
            if set_factor != 9999:
                factors = [set_factor]
            else:
                if whole_or_half: factors = [10, 1, 5,            0.1,                 0.5, 0.01,                    0.05]
                else:             factors = [10, 1, 2, 2.5, 3, 5, 0.1, 0.2, 0.25, 0.3, 0.5, 0.01, 0.02, 0.025, 0.03, 0.05]
                if whole_only:    factors = [10, 1,               0.1,                      0.01]

            
            # If we set a base_power_10, we use that...
            if set_base_power_10 != 9999:
                base_power_10 = set_base_power_10
            # otherwise, we find the best one.
            else:
                # We now get the largest absolute value between vmin and vmax...
                max_abs_tks = np.max([np.abs(_) for _ in tks])
                # and we find the power of 10 that is just below it...
                base_power_10 = 10**math.floor(np.log10(max_abs_tks))
                # so we ensure if it is equal to the value itself, we take the next lowest one.
                if base_power_10 == max_abs_tks: base_power_10 /= 10
            

            # We generate all the lists of ticks for all factors.
            tks_try = []
            tks_ignored_in_tkss_try = []
            for factor in factors:
                tks_ignored_in_tkss_try_i = []
                tks_try_i, step_i = generate_multiples(vmin, vmax, base_power_10, factor, center0)
                
                l_vminmax = np.abs(vmax-vmin)
                #if (len(tks_try_i) != 2) and (np.abs(tks_try_i[ 1]-vmin) < l_vminmax*factor_cut): tks_try_i.remove(tks_try_i[ 1])
                #if (len(tks_try_i) != 2) and (np.abs(tks_try_i[-2]-vmax) < l_vminmax*factor_cut): tks_try_i.remove(tks_try_i[-2])
                if (len(tks_try_i) != 2) and (np.abs(tks_try_i[ 1]-vmin) < l_vminmax*factor_cut): tks_ignored_in_tkss_try_i.append(tks_try_i[ 1])
                if (len(tks_try_i) != 2) and (np.abs(tks_try_i[-2]-vmax) < l_vminmax*factor_cut): tks_ignored_in_tkss_try_i.append(tks_try_i[-2])
                
                tks_try.append(tks_try_i)
                tks_ignored_in_tkss_try.append(tks_ignored_in_tkss_try_i)

            # Pick the best one based on the number of ticks.
            tks_try_respecting_limits = []
            tks_ignored_in_tkss_try_respecting_limits = []
            for iii0, tks_try_i in enumerate(tks_try):
                if number_of_tks_clean_min+2 <= len(tks_try_i) <= number_of_tks_clean_max+2:
                    tks_try_respecting_limits.append(tks_try_i)
                    tks_ignored_in_tkss_try_respecting_limits.append(tks_ignored_in_tkss_try[iii0])
            
            if len(tks_try_respecting_limits) == 0:
                tks = tks_try[0]
                tks_ignored_in_tkss = tks_ignored_in_tkss_try[0]
            else:
                if pick_clean_biggest_order_over_mean:
                    tks                 = tks_try_respecting_limits[                0]
                    tks_ignored_in_tkss = tks_ignored_in_tkss_try_respecting_limits[0]
                else:
                    tks                 = tks_try_respecting_limits[                int(round(np.mean(len(tks_try_respecting_limits)),0))]
                    tks_ignored_in_tkss = tks_ignored_in_tkss_try_respecting_limits[int(round(np.mean(len(tks_try_respecting_limits)),0))]

   
    
    else:

        # The goal here is to split the interval in two if it contains both positive and negative values and treat each interval as in the log_lin=False case when taking
        #    the log of the ends. We keep the log_min parameter as the limit towards 0 for the log scale.
        if vmin == 0: vmin = log_min
        if vmax == 0: vmin = log_max
        sgns = [np.sign(vmin), np.sign(vmax)]
        vmin_log = np.log10(np.abs(vmin)); vmax_log = np.log10(np.abs(vmax)); log_min_log = np.log10(np.abs(log_min))
        
        
        if sgns[0] == sgns[1]:
            log_min = np.min([np.abs(vmin), np.abs(vmax)])
        
        else:
            if auto_log_min:
                if (auto_log_min_equalityallowed and (np.abs(vmin) < log_min) and (np.abs(vmax) < log_min)) or (not auto_log_min_equalityallowed and (np.abs(vmin) <= log_min) and (np.abs(vmax) <= log_min)):
                    # We now get the smallest power of vmin and vmax...
                    min_tks_log = np.min([vmin_log, vmax_log])
                    # and we find the power of 10 that is just below it...
                    log_min = 10**math.floor(min_tks_log)
                    # so we ensure if it is equal to the value itself, we take the next lowest one.
                    if (not auto_log_min_equalityallowed) and log_min == round(10**min_tks_log,13): log_min /= 10
                        
            intervals = [[vmin, -log_min], [log_min, vmax]]
            if vmin >= -log_min: intervals[0] = [vmin]
            if vmax <=  log_min: intervals[1] = [vmax]
            if factor_cut_2intervals and (len(intervals[0]) + len(intervals[1]) == 4): factor_cut *= 2
            


        tks = []
        for i00, interval in enumerate(intervals):
            for ii0 in interval:  tks.append(ii0)
            if len(interval) != 1:
                tks_log_i = [np.log10(np.abs(interval[0])), np.log10(np.abs(interval[1]))]; tks_log_i.sort()
                if np.abs(tks_log_i[1]-tks_log_i[0]) >= 1:
                    #tks_i_fin = set_ticks(tks_log_i[0], tks_log_i[1], False, clean, d_tks_notclean, number_of_tks_notclean, prioritize0_lin_notclean, factor_cut, factor_cut_2intervals,
                    #                      number_of_tks_clean_min, number_of_tks_clean_max, pick_clean_biggest_order_over_mean,
                    #                      whole_or_half, True, set_factor, set_base_power_10, center0, log_min, auto_log_min, auto_log_min_equalityallowed,
                    #                      log_middle_include_0, only_tick_at_log_min, True,
                    #                      auto_stable_exponent, float_asitis_ends, g, stable_exponent, pure_1, pure_0, float_asitis, float_g, int_if_possible, int_base_if_possible, exponent_0, clean_powers)
                    log_span_integer_ticks = math.floor(tks_log_i[1]) - math.ceil(tks_log_i[0]) + 1
                    
                    if (set_factor == 9999) and (log_span_integer_ticks <= number_of_tks_clean_max+2):
                        set_factor_i = 1
                        set_base_power_10_i = 1
                    else:
                        set_factor_i = set_factor
                        set_base_power_10_i = set_base_power_10
                    
                    tks_i_fin = set_ticks(tks_log_i[0], tks_log_i[1], False, clean, d_tks_notclean, number_of_tks_notclean, prioritize0_lin_notclean, factor_cut, factor_cut_2intervals,
                                          number_of_tks_clean_min, number_of_tks_clean_max, pick_clean_biggest_order_over_mean,
                                          whole_or_half, True, set_factor_i, set_base_power_10_i, center0, log_min, auto_log_min, auto_log_min_equalityallowed,
                                          log_middle_include_0, only_tick_at_log_min, True,
                                          ignore_tks_in_tkss,
                                          auto_stable_exponent, float_asitis_ends, g, stable_exponent, pure_1, pure_0, float_asitis, float_g, int_if_possible, int_base_if_possible, exponent_0, clean_powers)
                    for _ in tks_i_fin[1:-1]:
                        tks.append(sgns[i00]*round(10**float(_),11))
                else:
                    tks_log_i = [np.abs(interval[0]), np.abs(interval[1])]; tks_log_i.sort()
                    tks_i_fin = set_ticks(tks_log_i[0], tks_log_i[1], log_lin=False, inside_log_use=True)
                    for _ in tks_i_fin[1:-1]:
                        tks.append(sgns[i00]*round(float(_),11))
                        ignored_tk = sgns[i00]*round(float(_),11)
                        if np.log10(np.abs(ignored_tk)) % 1 != 0:
                            tks_ignored_in_tkss.append(ignored_tk)

    tks = list(set(tks)); tks.sort()
    if log_lin:
        if log_middle_include_0 and len(intervals)==2: tks.append(0)
        tks = list(set(tks)); tks.sort()

    if inside_log_use: return tks
    
    if auto_stable_exponent: tkss, stable_exponent = latex_float_list(tks, auto_stable_exponent, float_asitis_ends, g, stable_exponent, pure_1, pure_0, float_asitis, float_g, int_if_possible, int_base_if_possible, exponent_0, clean_powers)
    else:                    tkss                  = latex_float_list(tks, auto_stable_exponent, float_asitis_ends, g, stable_exponent, pure_1, pure_0, float_asitis, float_g, int_if_possible, int_base_if_possible, exponent_0, clean_powers)
    
    if (log_lin and only_tick_at_log_min and (0 in tks)):
        if  log_min in tks: tkss[tks.index( log_min)] = ''
        if -log_min in tks: tkss[tks.index(-log_min)] = ''


    # If we chose to remove all tick labels in this case, we do so.
    if ignore_tks_in_tkss:
        for ignored_tk in tks_ignored_in_tkss:
            for i in range(len(tks)):
                if tks[i] == ignored_tk: tkss[i] = " "; break

    # Regardless, we still have to make sure the tkss do not overlap.
    for i0, interval in enumerate(intervals):
        interval_0 = interval[0]; interval_1 = interval[1]
        if log_lin: interval_0 = np.log10(np.abs(interval[0])); interval_1 = np.log10(np.abs(interval[1]))
        interval_cut = np.abs(interval_1 - interval_0)*factor_cut
        
        for i1 in range(len(tks)):
            if tkss[i1] != " ":
                # Finding out which tk is in this interval (and is an ignored one).
                if round(interval[0],14) < round(tks[i1],14) < round(interval[1],14):
                    tks_01 = tks[i1]
                    
                    if log_lin: tks_01 = np.log10(np.abs(tks[i1]))
    
                    if   np.abs(tks_01-interval_0) < interval_cut: tkss[i1] = " "
                    elif np.abs(tks_01-interval_1) < interval_cut: tkss[i1] = " "

    if auto_stable_exponent: return tks, tkss, stable_exponent
    else:                    return tks, tkss

---

In [ ]:
def generate_multiples(vmin, vmax, base_power_10, factor, center0=0):
    
    step = base_power_10 * factor

    # We begin from center0 and add multiples by factor of base_power_10.
    # Right now, we are not worried if these elements are in the [vmin, vmax] interval.
    tks_i = [center0]
    step_multiplier = 1
    while tks_i[-1] <= vmax:
        tks_i.append(center0 + step*step_multiplier)
        step_multiplier += 1
    step_multiplier = 1
    while tks_i[-1] >= vmin:
        tks_i.append(center0 - step*step_multiplier)
        step_multiplier += 1

    if vmin not in tks_i: tks_i.append(vmin)
    if vmax not in tks_i: tks_i.append(vmax)
    tks_i = [round(_,14) for _ in tks_i if vmin <= _ <= vmax]; tks_i.sort()

    return tks_i, step

In [ ]:
def sig_float_str(f, g=2):

    '''
    Returns the significant figure float endpoints.
    '''
    
    return f"{float(f):.{g}g}"

In [ ]:
def latex_float_list(f_list, auto_stable_exponent=False, float_asitis_ends=False, g=2, stable_exponent=9999, pure_1=True, pure_0=True, float_asitis=False, float_g=False, int_if_possible=False, int_base_if_possible=True, exponent_0=False, clean_powers=True):
    
    '''
    ### INPUT
    f_list               - list of numbers to be transformed into strings
    auto_stable_exponent - whether we wish to automatically pick a good, middle-ground stable_exponent value
    float_asitis_ends    - We might like just the start and end values to be left untouched as floats, only rounded by g.
    g                    - number of rounding digits
    
    stable_exponent      - see documentaiton of latex_float
    pure_1               - ...
    pure_0               - ...
    float_asitis         - ...
    float_g              - ...
    int_if_possible      - ...
    int_base_if_possible - ...
    exponent_0           - ...
    clean_powers         - ...
    '''
    
    
    
    if auto_stable_exponent:
        f_exponents = [math.floor(np.log10(np.abs(f))) for f in f_list if (f != 0)]
        stable_exponent = round(np.mean(f_exponents))
    
    

    LF = []
    if float_asitis_ends:
        LF.append(sig_float_str(f_list[0], g))
        for f in f_list[1:-1]: LF.append(latex_float(f, stable_exponent, pure_1, pure_0, float_asitis, float_g, g, int_if_possible, int_base_if_possible, exponent_0, clean_powers))
        LF.append(sig_float_str(f_list[-1], g))
    else:
        for f in f_list:       LF.append(latex_float(f, stable_exponent, pure_1, pure_0, float_asitis, float_g, g, int_if_possible, int_base_if_possible, exponent_0, clean_powers))
    
    if auto_stable_exponent: return LF, stable_exponent
    else:                    return LF

In [ ]:
def latex_float(f, stable_exponent=9999, pure_1=True, pure_0=True, float_asitis=False, float_g=False, g=2, int_if_possible=False, int_base_if_possible=True, exponent_0=False, clean_powers=True):
    
    '''
    ### INPUT
    f                    - the number we want to transform into a string
    stable_exponent      - Divide it by this power of 10 | Please note that it is set from the get-go... That is, we return the float as
                              pre-divided by that power, as you can see the treatment in latex_float_list, so we do not return the exponent
                              here when we use this parameter.
    pure_1               - if it is a -1/1, leave it as such
    pure_0               - same as pure_1 but for 0, instead of 0.0
    float_asitis         - Give it as as float with no rounding.
    float_g              - Give it as as float with    rounding given by g.
    g                    - number of rounding digits
    int_if_possible      - If the number is an integer, return it as such (not a float).
    int_base_if_possible - If we do want it as a float, but not as a float_asitis, then we get a base and an exponent. We might then want the
                              base to be an integer if possible. (The exponent always is an integer: exponent = int(exponent).)
    exponent_0           - If the exponent is 0, we might still want it as such.
    clean_powers         - If the result is a clean power of 10, we leave it without a product of 1 in front of it.
    '''



    if stable_exponent != 9999: f = round(f/10**stable_exponent,14)
    
    if pure_1: 
        if f == -1: return '-1'
        if f ==  1: return  '1'
    if pure_0:
        if f ==  0: return '0'
    
    if float_asitis:                         return str(float(f))
    if int_if_possible and (round(f) == f):  return str(int(f))
    if float_g or (stable_exponent != 9999): return str(round(float(f),g))

    # If it's not -1/1/0 and we do not want it float_asitis nor int_if_possible (using the set stable_exponent), then we treat it as a float
    #     and we round it by g. We get the base and the exponent from now on.
    globals_dict = {}
    exec('float_string = "{0:.'+str(g)+'e}".format('+str(f)+')', globals_dict)
    base, exponent = globals_dict.get('float_string', None).split("e")
    base = float(base); exponent = int(exponent)

    if int_base_if_possible and (round(base) == base): base = str(int(base))

    if not exponent_0 and (exponent == 0): return str(base)

    if clean_powers and (float(base) in [-1, 1]): return r"${0}0^{{{1}}}$".format(int(np.sign(float(base))), exponent)     
    else:                                         return r"${0} \times 10^{{{1}}}$".format(base, exponent)

---

In [ ]:
# A progress bar.

def progress_bar(current, total, bar_length=20):
    fraction = current / total

    arrow = int(fraction * bar_length - 1) * '-' + '>'
    padding = int(bar_length - len(arrow)) * ' '
    ending = '\n' if current == total else '\r'

    print(f'Progress: [{arrow}{padding}] {int(fraction*100)}%', end=ending)

---

In [ ]:
# For the colorbar formatting.

def sizeof_fmt(num, suffix='B'):
    ''' by Fred Cirera,  https://stackoverflow.com/a/1094933/1870254, modified'''
    for unit in ['','Ki','Mi','Gi','Ti','Pi','Ei','Zi']:
        if abs(num) < 1024.0:
            return "%3.1f %s%s" % (num, unit, suffix)
        num /= 1024.0
    return "%.1f %s%s" % (num, 'Yi', suffix)

---
---
---

### Turn png's into mp4.

In [ ]:
def get_indexed_pngs(folder: Path, prefix: str | None, extension: str):
    
    extension = extension if extension.startswith(".") else "." + extension

    def natural_key(path):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", path.name)]

    if prefix is None:
        files = [path for path in folder.iterdir() if path.suffix.lower() == extension.lower()]
        files.sort(key=natural_key)
        return files

    pattern = re.compile(rf"^{re.escape(prefix)}(\d+){re.escape(extension)}$", re.IGNORECASE)

    files = []
    for path in folder.iterdir():
        match = pattern.match(path.name)
        if match:
            index = int(match.group(1))
            files.append((index, path))
    
    files.sort(key=lambda x: x[0])
    return [path for _, path in files]

In [ ]:
def make_video(reverse_ORDER=False):
    
    output_folder = INPUT_FOLDER if OUTPUT_FOLDER is None else Path(OUTPUT_FOLDER)
    output_path   = output_folder / OUTPUT_NAME

    pngs = get_indexed_pngs(INPUT_FOLDER, PREFIX, EXTENSION)

    if reverse_ORDER:
        pngs = pngs[::-1]

    repeat_frames = DISPLAY_SECONDS * VIDEO_FPS
    
    if not pngs:                                         raise FileNotFoundError(f"No PNGs found matching: {PREFIX}<index>{EXTENSION}")
    if abs(repeat_frames - round(repeat_frames)) > 1e-9: raise ValueError(f"DISPLAY_SECONDS * VIDEO_FPS must be an integer. "f"Currently: {DISPLAY_SECONDS} * {VIDEO_FPS} = {repeat_frames}")
    repeat_frames = int(round(repeat_frames))

    sizes = []
    for path in pngs:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:                                  raise RuntimeError(f"Could not read image: {path}")
        sizes.append(img.shape[:2])

    height = max(h for h, w in sizes)
    width  = max(w for h, w in sizes)

    fourcc = cv2.VideoWriter_fourcc(*CODEC)
    writer = cv2.VideoWriter(str(output_path), fourcc, VIDEO_FPS, (width, height))

    if not writer.isOpened():                            raise RuntimeError("Could not open video writer.")

    for path in pngs:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)

        if img is None:                                  raise RuntimeError(f"Could not read image: {path}")

        img_height, img_width = img.shape[:2]

        if (img_width, img_height) != (width, height):
            
            pad_left   = (width  - img_width)  // 2
            pad_right  =  width  - img_width   - pad_left
            pad_top    = (height - img_height) // 2
            pad_bottom =  height - img_height  - pad_top

            img = cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_REPLICATE)

        for _ in range(repeat_frames): writer.write(img)

    writer.release()
    total_duration = len(pngs) * DISPLAY_SECONDS

    print(f"Saved video to: {output_path}")
    print(f"Images used: {len(pngs)}")
    print(f"Resolution: {width} x {height}")
    print(f"FPS: {VIDEO_FPS}")
    print(f"Each image duration: {DISPLAY_SECONDS} s")
    print(f"Total video duration: {total_duration:.2f} s")

---
---
---

In [ ]:
def rotation_matrix_x(angle_deg):
    
    t = np.deg2rad(angle_deg)
    c, s = np.cos(t), np.sin(t)
    
    return np.array([[1, 0,  0],
                     [0, c, -s],
                     [0, s,  c]])

In [ ]:
def rotation_matrix_y(angle_deg):
    
    t = np.deg2rad(angle_deg)
    c, s = np.cos(t), np.sin(t)
    
    return np.array([[ c, 0, s],
                     [ 0, 1, 0],
                     [-s, 0, c]])

In [ ]:
def rotation_matrix_z(angle_deg):
    
    t = np.deg2rad(angle_deg)
    c, s = np.cos(t), np.sin(t)
    
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]])

In [ ]:
def rotation_matrix_xyz(rx_deg, ry_deg, rz_deg):

    Rx = rotation_matrix_x(rx_deg)
    Ry = rotation_matrix_y(ry_deg)
    Rz = rotation_matrix_z(rz_deg)
    
    return Rz @ Ry @ Rx

---

In [ ]:
def make_ellipsoid_mesh_poles_along_A(a, b, c, nu=120, nv=60):
    
    # x = a cos(v), y = b sin(v) cos(u), z = c sin(v) sin(u)
    u = np.linspace(0, 2*np.pi, nu)
    v = np.linspace(0,   np.pi, nv)
    U, V = np.meshgrid(u, v)

    X = a * np.cos(V)
    Y = b * np.sin(V) * np.cos(U)
    Z = c * np.sin(V) * np.sin(U)

    return X, Y, Z

In [ ]:
def rotate_mesh(X, Y, Z, R):
    
    pts = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=0)
    pts_rot = R @ pts
    
    return pts_rot[0].reshape(X.shape), pts_rot[1].reshape(X.shape), pts_rot[2].reshape(X.shape)

In [ ]:
def draw_principal_axis(ax, p1, p2, color, lw=2.0, alpha=1.0, zorder=1000):
    
    ax.plot([p1[0], p2[0]],
            [p1[1], p2[1]],
            [p1[2], p2[2]], color=color, linewidth=lw, alpha=alpha, zorder=zorder)

In [ ]:
def get_view_direction(elev_deg, azim_deg):
    
    elev = np.deg2rad(elev_deg)
    azim = np.deg2rad(azim_deg)

    vx = np.cos(elev) * np.cos(azim)
    vy = np.cos(elev) * np.sin(azim)
    vz = np.sin(elev)

    v = np.array([vx, vy, vz], dtype=float)
    v /= np.linalg.norm(v)
    
    return v

---

In [ ]:
def make_surface_facecolors(X, Y, Z, a, b, c, R_obj, view_dir, base_color="#F4C56A", base_alpha=0.34):
    
    Nx = X / (a*a)
    Ny = Y / (b*b)
    Nz = Z / (c*c)

    N = np.stack([Nx.ravel(), Ny.ravel(), Nz.ravel()], axis=0)
    N /= np.linalg.norm(N, axis=0, keepdims=True)

    Nr = R_obj @ N
    Nxr = Nr[0].reshape(X.shape)
    Nyr = Nr[1].reshape(X.shape)
    Nzr = Nr[2].reshape(X.shape)

    facing = Nxr * view_dir[0] + Nyr * view_dir[1] + Nzr * view_dir[2]
    facing = np.clip(facing, 0.0, 1.0)

    brightness = 0.35 + 0.75 * facing
    alpha = base_alpha * (0.55 + 0.45 * facing)

    base_rgb = np.array(to_rgb(base_color))[None, None, :]
    rgb = np.clip(base_rgb * brightness[..., None], 0, 1)

    facecolors = np.zeros(X.shape + (4,))
    facecolors[..., :3] = rgb
    facecolors[..., 3] = alpha

    return facecolors

In [ ]:
def segment_is_front(p_obj_mid, a, b, c, R_obj, view_dir):
    
    # front/back test from outward normal at segment midpoint
    n_obj = np.array([p_obj_mid[0] / (a*a),
                      p_obj_mid[1] / (b*b),
                      p_obj_mid[2] / (c*c)], dtype=float)

    n_obj /= np.linalg.norm(n_obj)
    n_rot = R_obj @ n_obj

    return np.dot(n_rot, view_dir) >= 0.0

In [ ]:
def collect_wireframe_segments(X, Y, Z, Xr, Yr, Zr, a, b, c, R_obj, view_dir, rcount=18, ccount=28):
    
    segments = []

    nrows, ncols = X.shape
    row_idx = np.unique(np.round(np.linspace(0, nrows - 1, min(rcount, nrows))).astype(int))
    col_idx = np.unique(np.round(np.linspace(0, ncols - 1, min(ccount, ncols))).astype(int))

    # constant-v lines
    for i in row_idx:
        for j in range(ncols - 1):
            p1_obj = np.array([X[i, j],   Y[i, j],   Z[i, j]])
            p2_obj = np.array([X[i, j+1], Y[i, j+1], Z[i, j+1]])

            p1_rot = np.array([Xr[i, j],   Yr[i, j],   Zr[i, j]])
            p2_rot = np.array([Xr[i, j+1], Yr[i, j+1], Zr[i, j+1]])

            pm_obj = 0.5 * (p1_obj + p2_obj)
            pm_rot = 0.5 * (p1_rot + p2_rot)

            front = segment_is_front(pm_obj, a, b, c, R_obj, view_dir)
            depth = np.dot(pm_rot, view_dir)

            segments.append({"x": [p1_rot[0], p2_rot[0]],
                             "y": [p1_rot[1], p2_rot[1]],
                             "z": [p1_rot[2], p2_rot[2]],
                             "front": front,
                             "depth": depth})

    # constant-u lines
    for j in col_idx:
        for i in range(nrows - 1):
            p1_obj = np.array([X[i, j],   Y[i, j],   Z[i, j]])
            p2_obj = np.array([X[i+1, j], Y[i+1, j], Z[i+1, j]])

            p1_rot = np.array([Xr[i, j],   Yr[i, j],   Zr[i, j]])
            p2_rot = np.array([Xr[i+1, j], Yr[i+1, j], Zr[i+1, j]])

            pm_obj = 0.5 * (p1_obj + p2_obj)
            pm_rot = 0.5 * (p1_rot + p2_rot)

            front = segment_is_front(pm_obj, a, b, c, R_obj, view_dir)
            depth = np.dot(pm_rot, view_dir)

            segments.append({"x": [p1_rot[0], p2_rot[0]],
                             "y": [p1_rot[1], p2_rot[1]],
                             "z": [p1_rot[2], p2_rot[2]],
                             "front": front,
                             "depth": depth})

    return segments

In [ ]:
def draw_wireframe_subset(ax, segments, front_subset, color, linewidth, alpha, zorder_base):
    
    subset = [s for s in segments if s["front"] == front_subset]
    subset.sort(key=lambda s: s["depth"])   # far -> near inside each subset

    for k, s in enumerate(subset):
        ax.plot(s["x"], s["y"], s["z"],
                color=color,
                linewidth=linewidth,
                alpha=alpha,
                solid_capstyle="round",
                zorder=zorder_base + k)

---
---
---